### Data Ingestion

In [1]:
### document structure

from langchain_core.documents import Document

In [2]:
## PDF loader

from langchain_community.document_loaders import PyMuPDFLoader


/Users/hwct63@durham.ac.uk/Documents/Codes/LangDoc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = PyMuPDFLoader("../data/Paris_agreement-English.pdf")
document=loader.load()
print(document)

[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows)', 'creator': 'Adobe Acrobat Pro 11.0.0', 'creationdate': '2016-03-16T16:39:21-04:00', 'source': '../data/Paris_agreement-English.pdf', 'file_path': '../data/Paris_agreement-English.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': 'Paris Agreement English', 'author': 'UNFCCC', 'subject': 'Paris Agreement English', 'keywords': '', 'moddate': '2016-03-18T13:54:30+01:00', 'trapped': '', 'modDate': "D:20160318135430+01'00'", 'creationDate': "D:20160316163921-04'00'", 'page': 0}, page_content='PARIS AGREEMENT \n(mm \nUNITED NATIONS \n2015'), Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows)', 'creator': 'Adobe Acrobat Pro 11.0.0', 'creationdate': '2016-03-16T16:39:21-04:00', 'source': '../data/Paris_agreement-English.pdf', 'file_path': '../data/Paris_agreement-English.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': 'Paris Agreement English', 'author': 'UNFCCC', 'subject': 'Paris Agreement English', 

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   
from pathlib import Path

In [5]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 393 PDF files to process

Processing: FINAL UPDATED NAMIBIA NDC 2023.pdf
  ✓ Loaded 44 pages

Processing: Ghana's Updated Nationally Determined Contribution to the UNFCCC_2021.pdf
  ✓ Loaded 27 pages

Processing: UK's 2035 NDC ICTU.pdf
  ✓ Loaded 74 pages

Processing: Mozambique ProvNDC_ENG.pdf
  ✓ Loaded 25 pages

Processing: 2019.09.19_DPRK letter to SG special envoy for NDC.pdf
  ✓ Loaded 3 pages

Processing: NDC_TAJIKISTAN_RUSS.pdf
  ✓ Loaded 37 pages

Processing: Uzbekistan_Updated NDC_2021_RU.pdf
  ✓ Loaded 31 pages

Processing: NDC3.0_Kyrgyzstan_English_30-09-2025 (2).pdf
  ✓ Loaded 94 pages

Processing: Cuban First NDC Summary (Updated submission).pdf
  ✓ Loaded 6 pages

Processing: 202203111154---KSA NDC 2021.pdf
  ✓ Loaded 12 pages

Processing: 211223_The Republic of Korea's Enhanced Update of its First Nationally Determined Contribution_211227_editorial change.pdf
  ✓ Loaded 30 pages

Processing: NDC_RF_eng.pdf
  ✓ Loaded 19 pages

Processing: Submission  UPDATE  NDC.p

In [6]:
all_pdf_documents


[Document(metadata={'producer': '3-Heights™ PDF Merge Split Shell 6.12.1.11 (http://www.pdf-tools.com)', 'creator': '', 'creationdate': '', 'source': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_path': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'total_pages': 44, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-01-11T11:35:47+00:00', 'trapped': '', 'modDate': 'D:20240111113547Z', 'creationDate': '', 'page': 0, 'source_file': 'FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNamibia’s Nationally \nDetermined Contribution \n2023 • SECOND UPDATE'),
 Document(metadata={'producer': '3-Heights™ PDF Merge Split Shell 6.12.1.11 (http://www.pdf-tools.com)', 'creator': '', 'creationdate': '', 'source': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_path': '../data/FINAL UPDATED NAMIBIA N

In [7]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks=split_documents(all_pdf_documents)
chunks

Split 16760 documents into 53246 chunks

Example chunk:
Content: 1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Namibia’s Nationally 
Determined Contribution 
2023 • SECOND UPDATE...
Metadata: {'producer': '3-Heights™ PDF Merge Split Shell 6.12.1.11 (http://www.pdf-tools.com)', 'creator': '', 'creationdate': '', 'source': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_path': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'total_pages': 44, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-01-11T11:35:47+00:00', 'trapped': '', 'modDate': 'D:20240111113547Z', 'creationDate': '', 'page': 0, 'source_file': 'FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '3-Heights™ PDF Merge Split Shell 6.12.1.11 (http://www.pdf-tools.com)', 'creator': '', 'creationdate': '', 'source': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_path': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'total_pages': 44, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-01-11T11:35:47+00:00', 'trapped': '', 'modDate': 'D:20240111113547Z', 'creationDate': '', 'page': 0, 'source_file': 'FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNamibia’s Nationally \nDetermined Contribution \n2023 • SECOND UPDATE'),
 Document(metadata={'producer': '3-Heights™ PDF Merge Split Shell 6.12.1.11 (http://www.pdf-tools.com)', 'creator': '', 'creationdate': '', 'source': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_path': '../data/FINAL UPDATED NAMIBIA N

### embedding and vectorstoreDB


In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
from pydantic_settings import BaseSettings
# import chromadb # Removed ChromaDB import
# from chromadb.config import Settings # Removed ChromaDB import
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import faiss # Added Faiss import
import os # Added os import for persistence

In [10]:
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "BAAI/bge-m3"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            self._embedding_dim = self.model.get_sentence_embedding_dimension()
            print(f"Model loaded successfully. Embedding dimension: {self._embedding_dim}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def get_embedding_dim(self) -> int:
        """Return the dimension of the embeddings"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self._embedding_dim

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        # Ensure embeddings are float32, which Faiss prefers
        embeddings = self.model.encode(texts, show_progress_bar=True).astype('float32')
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# Initialize the embedding manager with BGE-M3
embedding_manager = EmbeddingManager()

Loading embedding model: BAAI/bge-m3
Model loaded successfully. Embedding dimension: 1024


## vector store


In [11]:
import os
import pickle
import uuid
from typing import Any, Dict, List, Optional, Tuple

import faiss
import numpy as np


class FaissDocStore:
    """
    In-memory store for document text and metadata, mapped by Faiss internal index.
    Faiss itself only stores vectors and internal IDs.
    """

    def __init__(self):
        self.store: Dict[int, Dict[str, Any]] = {}  # Key: Faiss internal index -> {'text', 'metadata', 'original_id'}
        self.next_index = 0

    def add(self, text: str, metadata: Dict[str, Any], original_id: str) -> int:
        """Adds a document and returns the internal Faiss index."""
        current_index = self.next_index
        self.store[current_index] = {
            'text': text,
            'metadata': metadata,
            'original_id': original_id
        }
        self.next_index += 1
        return current_index

    def get_by_index(self, index: int) -> Dict[str, Any]:
        """Retrieves a document by its internal Faiss index."""
        return self.store.get(index)

    def count(self) -> int:
        """Returns the number of stored documents."""
        return len(self.store)

    def save(self, path: str):
        """Persist the docstore (text + metadata) to disk."""
        with open(path, "wb") as f:
            pickle.dump({"store": self.store, "next_index": self.next_index}, f)

    @classmethod
    def load(cls, path: str) -> "FaissDocStore":
        """Load a previously persisted docstore from disk."""
        instance = cls()
        with open(path, "rb") as f:
            data = pickle.load(f)
        instance.store = data["store"]
        instance.next_index = data["next_index"]
        return instance


class VectorStore:
    """Manages document embeddings in a Faiss vector store with a persisted document store."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.

        Args:
            collection_name: Used as the base filename for the saved index/docstore.
            persist_directory: Directory where the Faiss index + docstore are saved/loaded from.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.embedding_dim = embedding_manager.get_embedding_dim()
        self.index = None  # The Faiss index
        self.docstore = FaissDocStore()  # Separate store for text and metadata

        self._index_path = os.path.join(self.persist_directory, f"{self.collection_name}.faiss")
        self._docstore_path = os.path.join(self.persist_directory, f"{self.collection_name}_docstore.pkl")

        self._initialize_store()

    def _initialize_store(self):
        """
        Initialize Faiss index and document store.

        If a persisted store already exists at `persist_directory`, load it directly
        instead of starting from an empty index. This lets the app reuse an
        already-embedded collection without recomputing embeddings.
        """
        try:
            if self._exists_on_disk():
                print(f"Found existing vector store at '{self.persist_directory}', loading from disk...")
                self.load_local()
            else:
                # Faiss Index: IndexFlatL2 is a simple index for exact L2 (Euclidean) distance search.
                self.index = faiss.IndexFlatL2(self.embedding_dim)
                print(f"Vector store initialized (Faiss IndexFlatL2). Embedding dimension: {self.embedding_dim}")

            print(f"Existing documents in collection: {self.docstore.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def _exists_on_disk(self) -> bool:
        """Check whether a saved index + docstore are present for this collection."""
        return os.path.exists(self._index_path) and os.path.exists(self._docstore_path)

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store, then persist to disk.

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents (must be numpy float32)
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        if embeddings.dtype != np.float32:
            raise ValueError("Embeddings must be of type numpy.float32 for Faiss")

        print(f"Adding {len(documents)} documents to vector store...")

        try:
            current_count = self.index.ntotal
            self.index.add(embeddings)

            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{current_count + i}"

                metadata = dict(doc.metadata)
                metadata['doc_index_in_batch'] = i
                metadata['content_length'] = len(doc.page_content)

                self.docstore.add(doc.page_content, metadata, doc_id)

            if self.index.ntotal != self.docstore.count():
                raise Exception("Faiss index count and DocStore count mismatch after addition.")

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.index.ntotal}")

            # Persist immediately so this embedded stage can be reused later
            # (e.g. on the next run) without recomputing embeddings.
            self.save_local()

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

    def search_embeddings(self, query_embedding: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        """
        Search the Faiss index for nearest neighbors.

        Args:
            query_embedding: The query vector (1, embedding_dim)
            top_k: Number of results to return

        Returns:
            Tuple of (distances, indices)
        """
        D, I = self.index.search(query_embedding, top_k)
        return D[0], I[0]

    def save_local(self, persist_directory: Optional[str] = None):
        """
        Save the Faiss index and the docstore to disk so the vector store
        can be reloaded later without redoing the embedding step.
        """
        directory = persist_directory or self.persist_directory
        os.makedirs(directory, exist_ok=True)

        index_path = os.path.join(directory, f"{self.collection_name}.faiss")
        docstore_path = os.path.join(directory, f"{self.collection_name}_docstore.pkl")

        faiss.write_index(self.index, index_path)
        self.docstore.save(docstore_path)

        print(f"Vector store saved to '{directory}' "
              f"(index: '{os.path.basename(index_path)}', docstore: '{os.path.basename(docstore_path)}')")

    def load_local(self, persist_directory: Optional[str] = None):
        """
        Load a previously saved Faiss index and docstore from disk.
        """
        directory = persist_directory or self.persist_directory
        index_path = os.path.join(directory, f"{self.collection_name}.faiss")
        docstore_path = os.path.join(directory, f"{self.collection_name}_docstore.pkl")

        if not (os.path.exists(index_path) and os.path.exists(docstore_path)):
            raise FileNotFoundError(f"No saved vector store found at '{directory}'")

        self.index = faiss.read_index(index_path)
        self.docstore = FaissDocStore.load(docstore_path)

        print(f"Vector store loaded from '{directory}'. Total documents: {self.index.ntotal}")




In [12]:
# ---------------------------------------------------------------------------
# Usage: build the store once, reuse the embedded stage on later runs
# ---------------------------------------------------------------------------

vectorstore = VectorStore()

if vectorstore.index.ntotal == 0:
    # Nothing persisted yet (or the persisted store was empty) — embed from scratch.

    ### Convert the text to embeddings
    texts = [doc.page_content for doc in chunks]

    ## Generate the Embeddings
    embeddings = embedding_manager.generate_embeddings(texts)

    ## Store in the vector database (this also calls save_local() internally)
    vectorstore.add_documents(chunks, embeddings)
else:
    print(f"Using existing embedded vector store ({vectorstore.index.ntotal} documents) — skipping re-embedding.")

Vector store initialized (Faiss IndexFlatL2). Embedding dimension: 1024
Existing documents in collection: 0
Generating embeddings for 53246 texts...


Batches:   0%|          | 0/1664 [00:33<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
chunks


[Document(metadata={'producer': '3-Heights™ PDF Merge Split Shell 6.12.1.11 (http://www.pdf-tools.com)', 'creator': '', 'creationdate': '', 'source': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_path': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'total_pages': 44, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-01-11T11:35:47+00:00', 'trapped': '', 'modDate': 'D:20240111113547Z', 'creationDate': '', 'page': 0, 'source_file': 'FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNamibia’s Nationally \nDetermined Contribution \n2023 • SECOND UPDATE'),
 Document(metadata={'producer': '3-Heights™ PDF Merge Split Shell 6.12.1.11 (http://www.pdf-tools.com)', 'creator': '', 'creationdate': '', 'source': '../data/FINAL UPDATED NAMIBIA NDC 2023.pdf', 'file_path': '../data/FINAL UPDATED NAMIBIA N

In [ ]:
# ### Convert the text to embeddings
# texts=[doc.page_content for doc in chunks]

# ## Generate the Embeddings

# embeddings=embedding_manager.generate_embeddings(texts)

# ##store int he vector dtaabase
# vectorstore.add_documents(chunks,embeddings)



### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 52933 texts...


Batches: 100%|██████████| 1655/1655 [12:35<00:00,  2.19it/s]


Generated embeddings with shape: (52933, 384)
Adding 52933 documents to vector store...
Successfully added 52933 documents to vector store
Total documents in collection: 52933


In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        # Note: If the embeddings are L2-normalized (which SentenceTransformers often does by default),
        # L2 distance is directly related to Cosine Similarity by:
        # Cosine Sim = 1 - (L2_distance^2) / 2
        # We'll use this formula for a correct similarity score.

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold (0.0 to 1.0)
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # 1. Generate and Normalize Query Embedding
        # Faiss search requires float32. We rely on EmbeddingManager to handle this.
        # Ensure the query embedding is L2-normalized for the Cosine Sim formula to work.
        query_embedding_np = self.embedding_manager.generate_embeddings([query])
        # Manually L2-normalize if the SentenceTransformer model doesn't guarantee it, 
        # as L2 normalization is standard for cosine similarity via L2 distance.
        faiss.normalize_L2(query_embedding_np) 
        
        # 2. Search in vector store
        try:
            # Faiss search returns L2 distances (D) and internal indices (I)
            distances, indices = self.vector_store.search_embeddings(query_embedding_np, top_k)

            # 3. Process results
            retrieved_docs = []
            
            for i, (index, distance) in enumerate(zip(indices, distances)):
                # Faiss returns index -1 if not enough results are found
                if index == -1:
                    continue
                    
                # Get document and metadata from the separate DocStore
                doc_data = self.vector_store.docstore.get_by_index(index)
                
                if doc_data:
                    
                    # 4. Calculate Correct Cosine Similarity
                    # Use the relationship between L2 distance and Cosine Similarity for L2-normalized vectors.
                    # Cosine Sim = 1 - (L2_distance^2) / 2
                    # Note: L2 distances in Faiss are typically non-negative.
                    
                    # Clamp distances to ensure the result is in [0, 1] for robustness
                    clamped_distance_sq = np.clip(distance ** 2, 0.0, 2.0)
                    
                    similarity_score = 1.0 - (clamped_distance_sq / 2.0)
                    
                    # 5. Apply Score Threshold (minimum similarity)
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_data['original_id'],
                            'content': doc_data['text'],
                            'metadata': doc_data['metadata'],
                            'similarity_score': float(similarity_score), # Convert numpy float to native float
                            'distance': float(distance), # Faiss L2 distance
                            'rank': i + 1
                        })
            
            print(f"Retrieved {len(retrieved_docs)} documents (after filtering with score >= {score_threshold})")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

# Ensure the faiss.normalize_L2 import is available at the top of the file
# and the VectorStore initialization uses the EmbeddingManager to get the dimension.

In [ ]:
import os
from typing import List, Dict, Any
import numpy as np
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder


# ==========================================
# 1. LLM Client Wrapper (HyDE Generation)
# ==========================================
from ollama import Client


class LLMClientWrapper:
    """Wrapper for LLM text generation using Ollama Cloud API."""

    def __init__(
        self,
        model_name: str = "gpt-oss:120b-cloud",
        host: str = None,
        api_key: str = None,
    ):
        """
        Args:
            model_name: The Ollama cloud model name (e.g., 'gpt-oss:120b-cloud', 'deepseek-v3.1:671b-cloud').
            host: Endpoint URL. Defaults to 'https://ollama.com' for Ollama Cloud.
            api_key: Ollama API key for cloud authentication (falls back to OLLAMA_API_KEY env var).
        """
        self.model_name = model_name

        # Resolve credentials and target host
        resolved_api_key = api_key or os.getenv("OLLAMA_API_KEY")
        self.host = host or os.getenv("OLLAMA_HOST", "https://ollama.com")

        # Set Bearer token header for Cloud API access
        headers = {}
        if resolved_api_key:
            headers["Authorization"] = f"Bearer {resolved_api_key}"

        self.client = Client(host=self.host, headers=headers)

    def generate_text(self, prompt: str) -> str:
        """
        Generates text using Ollama Cloud for HyDE hypothetical document creation.

        Args:
            prompt: Prompt requesting the ideal theoretical answer.

        Returns:
            Generated text string.
        """
        response = self.client.generate(
            model=self.model_name,
            prompt=prompt,
            options={
                "temperature": 0.7,
                "num_predict": 300,  # Ollama parameter for max output tokens
            },
        )
        return response["response"].strip()

# ==========================================
# 2. Sparse Retriever Wrapper (BM25 Keyword Search)
# ==========================================
class BM25SparseRetriever:
    """Sparse keyword search retriever built on BM25Okapi."""
    
    def __init__(self, documents: List[Dict[str, Any]]):
        """
        Initialize and index documents with BM25.
        
        Args:
            documents: List of dicts containing document data:
                       [{'original_id': 'doc_1', 'text': '...', 'metadata': {...}}, ...]
        """
        self.documents = documents
        # Simple whitespace tokenization & lowercasing for BM25 indexing
        self.corpus_tokens = [doc['text'].lower().split() for doc in documents]
        self.bm25 = BM25Okapi(self.corpus_tokens)

    def search(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Performs BM25 keyword matching against the indexed corpus.
        
        Args:
            query: Raw user search query (e.g., exact product codes, names).
            top_k: Number of candidate documents to return.
            
        Returns:
            List of matching document dicts formatted for the hybrid pipeline.
        """
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)
        
        # Get top-k indices sorted by score descending
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        
        results = []
        for idx in top_indices:
            # Filter out non-matching documents with a score of 0
            if scores[idx] > 0.0:
                doc = self.documents[idx].copy()
                doc['bm25_score'] = float(scores[idx])
                results.append(doc)
                
        return results


# ==========================================
# 3. Cross-Encoder Wrapper (Reranker)
# ==========================================
class CrossEncoderWrapper:
    """Wrapper for sentence-transformers CrossEncoder models."""
    
    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        """
        Args:
            model_name: HuggingFace model checkpoint for reranking.
        """
        self.model = CrossEncoder(model_name)

    def predict(self, pairs: List[List[str]]) -> np.ndarray:
        """
        Evaluates relevance scores for query-document pairs.
        
        Args:
            pairs: List of [query, document_text] pairs.
            
        Returns:
            Numpy array of cross-encoder logit relevance scores.
        """
        return self.model.predict(pairs)

In [ ]:
import numpy as np
import faiss
from typing import List, Dict, Any

class AdvancedRAGRetriever:
    """Handles multi-stage query retrieval utilizing HyDE, Hybrid Search, and Cross-Encoder Reranking."""
    
    def __init__(
        self, 
        vector_store: Any, 
        embedding_manager: Any,
        llm_client: Any,
        sparse_retriever: Any,
        cross_encoder: Any
    ):
        """
        Initialize the advanced retriever
        
        Args:
            vector_store: Vector store containing document embeddings (FAISS)
            embedding_manager: Manager for generating query embeddings
            llm_client: Client to generate theoretical answers for HyDE (e.g., OpenAI API wrapper)
            sparse_retriever: Keyword search component (e.g., BM25)
            cross_encoder: Reranker model (e.g., sentence_transformers.CrossEncoder)
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.llm_client = llm_client
        self.sparse_retriever = sparse_retriever
        self.cross_encoder = cross_encoder

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -10.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents using HyDE -> Hybrid Search -> Reranking
        """
        print(f"Retrieving documents for query: '{query}'")
        
        # We fetch more candidates initially to give the reranker a good pool
        fetch_k = top_k * 3 

        # ==========================================
        # STAGE 1: HyDE (Hypothetical Document Embeddings)
        # ==========================================
        # Ask the LLM to generate a theoretical perfect answer to the user's query
        hyde_prompt = f"Please write a brief, authoritative passage that perfectly answers the following query: {query}"
        try:
            hypothetical_answer = self.llm_client.generate_text(hyde_prompt)
        except Exception as e:
            print(f"HyDE generation failed, falling back to original query. Error: {e}")
            hypothetical_answer = query

        # Embed the hypothetical answer instead of the raw query
        query_embedding_np = self.embedding_manager.generate_embeddings([hypothetical_answer])
        faiss.normalize_L2(query_embedding_np) 

        # ==========================================
        # STAGE 2: Hybrid Search (Dense + Sparse)
        # ==========================================
        merged_candidates: Dict[str, Dict[str, Any]] = {}

        # 2A. Dense Retrieval (using HyDE embedding)
        try:
            distances, indices = self.vector_store.search_embeddings(query_embedding_np, fetch_k)
            for index, distance in zip(indices[0], distances[0]):
                if index == -1: continue
                doc_data = self.vector_store.docstore.get_by_index(index)
                if doc_data:
                    doc_id = doc_data['original_id']
                    merged_candidates[doc_id] = {
                        'id': doc_id,
                        'content': doc_data['text'],
                        'metadata': doc_data['metadata'],
                        'source': 'dense'
                    }
        except Exception as e:
            print(f"Dense retrieval error: {e}")

        # 2B. Sparse Retrieval (using Original Keyword Query)
        # Note: BM25/SPLADE is better suited for the raw user query to catch exact product codes
        try:
            sparse_results = self.sparse_retriever.search(query, fetch_k)
            for doc in sparse_results:
                doc_id = doc['original_id']
                if doc_id not in merged_candidates:
                    merged_candidates[doc_id] = {
                        'id': doc_id,
                        'content': doc['text'],
                        'metadata': doc['metadata'],
                        'source': 'sparse'
                    }
                else:
                    merged_candidates[doc_id]['source'] = 'hybrid' # Found in both
        except Exception as e:
            print(f"Sparse retrieval error: {e}")

        candidate_list = list(merged_candidates.values())
        if not candidate_list:
            return []

        # ==========================================
        # STAGE 3: Cross-Encoder Reranking
        # ==========================================
        # Create (Query, Document) pairs. We use the original query, NOT the HyDE answer, 
        # so the reranker strictly evaluates relevance to what the user actually asked.
        pairs = [[query, doc['content']] for doc in candidate_list]
        
        try:
            # Reranker returns logits/scores for each pair
            rerank_scores = self.cross_encoder.predict(pairs)
            
            # Attach scores to documents
            for doc, score in zip(candidate_list, rerank_scores):
                doc['cross_encoder_score'] = float(score)
                
            # Sort by highest score first
            candidate_list.sort(key=lambda x: x['cross_encoder_score'], reverse=True)
            
            # Filter by threshold and truncate to top_k
            final_docs = []
            for rank, doc in enumerate(candidate_list):
                if doc['cross_encoder_score'] >= score_threshold:
                    doc['rank'] = rank + 1
                    final_docs.append(doc)
                    
                if len(final_docs) == top_k:
                    break
                    
            print(f"Retrieved {len(final_docs)} documents after reranking.")
            return final_docs

        except Exception as e:
            print(f"Reranking error: {e}")
            # Fallback: return the unranked merged candidates up to top_k
            return candidate_list[:top_k]

# Initialization example
# rag_retriever = AdvancedRAGRetriever(
#     vector_store=vectorstore,
#     embedding_manager=embedding_manager,
#     llm_client=my_openai_wrapper,
#     sparse_retriever=my_bm25_retriever,
#     cross_encoder=CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
# )

In [ ]:
# class RAGRetriever:
#     """Handles query-based retrieval from the vector store"""
    
#     def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
#         """
#         Initialize the retriever
        
#         Args:
#             vector_store: Vector store containing document embeddings
#             embedding_manager: Manager for generating query embeddings
#         """
#         self.vector_store = vector_store
#         self.embedding_manager = embedding_manager
#         # The internal Faiss index uses L2 distance (Euclidean). 
#         # For 'all-MiniLM-L6-v2', L2 distance on normalized vectors is equivalent 
#         # to a linear transformation of cosine distance.
#         # Cosine Similarity = 1 - (L2_distance^2) / 2 for L2-normalized vectors.
#         # We will use the formula: Similarity = 1 / (1 + Distance) for a simple non-linear scaling of distance.
#         # OR we can compute the cosine similarity manually as Faiss only gives L2 distance for IndexFlatL2.
#         # We'll use the cosine_similarity function from sklearn on the retrieved embedding.

#     def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
#         """
#         Retrieve relevant documents for a query
        
#         Args:
#             query: The search query
#             top_k: Number of top results to return
#             score_threshold: Minimum similarity score threshold
            
#         Returns:
#             List of dictionaries containing retrieved documents and metadata
#         """
#         print(f"Retrieving documents for query: '{query}'")
#         print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
#         # Generate query embedding
#         query_embedding_np = self.embedding_manager.generate_embeddings([query])
        
#         # Search in vector store
#         try:
#             # Faiss search returns L2 distances and internal indices
#             distances, indices = self.vector_store.search_embeddings(query_embedding_np, top_k)

#             # Process results
#             retrieved_docs = []
            
#             for i, (index, distance) in enumerate(zip(indices, distances)):
#                 # Faiss returns index -1 if not enough results are found
#                 if index == -1:
#                     continue
                    
#                 # Get document and metadata from the separate DocStore
#                 doc_data = self.vector_store.docstore.get_by_index(index)
                
#                 if doc_data:
#                     # Cosine Similarity is often preferred for normalized sentence embeddings
#                     # For simplicity and to match the 'distance' concept, 
#                     # we will just invert the rank based on distance for now.
#                     # Note: L2 distance in Faiss is *not* directly cosine distance (which is what ChromaDB used).
#                     # A similarity score approximation (1 / (1 + D)) is simple, 
#                     # or we could normalize vectors before adding and then compute 
#                     # similarity from L2 distance (Sim = 1 - (D**2)/2).
#                     # For now, let's use the distance as the score for simplicity 
#                     # (lower distance = better match, which is opposite of similarity).
#                     # We will calculate a proper cosine similarity just for the score:
#                     # In a production system, you'd ensure vectors are normalized for cosine similarity.

#                     # For Faiss IndexFlatL2:
#                     # The distance is L2. For normalized vectors, Cosine Similarity = 1 - (L2_distance^2) / 2
                    
#                     # For RAG, we will keep the distance as is and use a score threshold on L2 distance
#                     # where lower is better (i.e., we treat score_threshold as max_distance).
                    
#                     # To align with the ChromaDB style (score_threshold is minimum *similarity*):
#                     # We will calculate the cosine similarity for the score
#                     # (this requires having the original vector, which Faiss doesn't store by default,
#                     # so we will use the simpler method of converting L2 distance to an approximate score
#                     # for this example, or just return L2 distance).
                    
#                     # Sticking to the L2 distance output:
                    
#                     # We will approximate similarity as: 1 / (1 + distance) 
#                     similarity_score = 1 / (1 + distance) 
                    
#                     # We have to reverse the threshold logic since the original code expected 
#                     # a minimum similarity score (higher is better). 
#                     # Our approximated similarity score is also higher=better.
                    
#                     if similarity_score >= score_threshold:
#                         retrieved_docs.append({
#                             'id': doc_data['original_id'],
#                             'content': doc_data['text'],
#                             'metadata': doc_data['metadata'],
#                             'similarity_score': similarity_score, # Approximate score
#                             'distance': distance, # Faiss L2 distance
#                             'rank': i + 1
#                         })
                
#                 print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
#             else:
#                 print("No documents found")
            
#             return retrieved_docs
            
#         except Exception as e:
#             print(f"Error during retrieval: {e}")
#             return []

# rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [ ]:
rag_retriever


In [ ]:
rag_retriever.retrieve("What is paris agreement")


Retrieving documents for query: 'What is paris agreement'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering with score >= 0.0)


[{'id': 'doc_45026ac3_35441',
  'content': 'PARIS AGREEMENT \n(mm \nUNITED NATIONS \n2015',
  'metadata': {'producer': 'Acrobat Distiller 11.0 (Windows)',
   'creator': 'Adobe Acrobat Pro 11.0.0',
   'creationdate': '2016-03-16T16:39:21-04:00',
   'source': '../data/Paris_agreement-English.pdf',
   'file_path': '../data/Paris_agreement-English.pdf',
   'total_pages': 27,
   'format': 'PDF 1.7',
   'title': 'Paris Agreement English',
   'author': 'UNFCCC',
   'subject': 'Paris Agreement English',
   'keywords': '',
   'moddate': '2016-03-18T13:54:30+01:00',
   'trapped': '',
   'modDate': "D:20160318135430+01'00'",
   'creationDate': "D:20160316163921-04'00'",
   'page': 0,
   'source_file': 'Paris_agreement-English.pdf',
   'file_type': 'pdf',
   'doc_index_in_batch': 35441,
   'content_length': 42},
  'similarity_score': 0.8996894955635071,
  'distance': 0.44790738821029663,
  'rank': 1},
 {'id': 'doc_9ce06f01_8005',
  'content': '(c)   Other contextual aspirations \nand priorities ac

### RAG pipeline to VectorDB o LLM Output Generation

import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import PromptTemplate

from langchain_classic.schema import HumanMessage, SystemMessage



In [ ]:
class OllamaLLM:
    # Removed api_key from __init__ since Ollama runs locally
    def __init__(self, model_name: str = "deepseek-r1:8b", host: str = "http://localhost:11434"):
        """
        Initialize Ollama LLM
        
        Args:
            model_name: Ollama model name (e.g., deepseek-r1:8b, llama3)
            host: Ollama server URL (default: http://localhost:11434)
        """
        self.model_name = model_name
        self.host = host
        
        # Ollama does not use an API key, so we remove the check
        
        # Instantiate ChatOllama
        self.llm = ChatOllama(
            model=self.model_name,
            temperature=0.1,
            # max_tokens is now num_predict in ChatOllama
            num_predict=8192,
            base_url=self.host
        )
        
        print(f"Initialized Ollama LLM with model: {self.model_name}")
        print(f"Make sure Ollama server is running at {self.host} and model is pulled.")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length (Note: ChatOllama uses num_predict for this)
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysis of the provided context to answer the user's question.

### Guidelines
1. **Unrestricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question."

### Response Structure
You must provide your answer in three distinct, detailed parts:

**Part 1: Comprehensive Analysis of the Agreement**
Identify, quote, and analyze *every* specific section from the provided agreement that is relevant to the question. Elaborate on the definitions and specific wording used in the text.

**Part 2: Relevant Legal Standards & Regulations**
Identify and detail the relevant sections from provided standards or legal materials (e.g., ISO, regulations). Explain the specific metrics, compliance requirements, or technical obligations in full.

**Part 3: In-Depth Legal Assessment**
Based on the comprehensive evidence gathered in Parts 1 and 2, provide a lengthy and detailed legal conclusion. Synthesize the information to explore the implications, obligations, and dispute resolution mechanisms fully.

### Context
{context}

### Question
{question}

### Answer
"""
)
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            # Note: Ollama models are typically sensitive to the message format (HumanMessage/SystemMessage).
            # We stick to the original LangChain message structure.
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [ ]:
# Initialize Ollama LLM 
try:
    # Use the target model name: deepseek-r1:8b
    ollama_llm = OllamaLLM(model_name="deepseek-r1:8b") 
    print("Ollama LLM initialized successfully!")
# Changed the error handling since api_key is no longer relevant
except Exception as e:
    print(f"Warning: Error initializing Ollama LLM: {e}")
    print("Please ensure Ollama is installed and running, and the model 'deepseek-r1:8b' is pulled.")
    ollama_llm = None

Initialized Ollama LLM with model: deepseek-r1:8b
Make sure Ollama server is running at http://localhost:11434 and model is pulled.
Ollama LLM initialized successfully!


In [ ]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering with score >= 0.0)


[{'id': 'doc_ee509ba3_37981',
  'content': 'identified in the previous NDC (NDC 2.0) and confirmed within national policies, \nprograms, and plans, this NDC refines those measures to enhance cross-sector \ncollaboration in planning and implementation. Greater attention has been given \nto strengthening governance to support the coordinated execution of these \nmeasures, which was a significant challenge in delivering the previous NDC. \nThe updated adaptation component also introduces an enhanced Monitoring, \nEvaluation, and Learning (MEL) framework, highlighting the critical role of \nlearning in adaptation. This involves analysing collected data and information to \ninform decision-making, generating knowledge about what has worked, what \nhas not, and determining which adaptation actions have achieved the expected \noutcomes. This process will also improve tracking and reporting of progress at \nboth national and international levels. The framework closely aligns with the',
  'meta

##Integration Vectordb Context pipeline With LLM output


In [ ]:
### Simple RAG pipeline with Ollama LLM
# from langchain_groq import ChatGroq
from langchain_community.chat_models import ChatOllama
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Ollama LLM (The model is deepseek-r1:8b)
# Removed groq_api_key logic
ollama_model_name = "deepseek-r1:8b" 

# Instantiate the Ollama LLM directly using the new class
llm=ChatOllama(model=ollama_model_name,temperature=0.1,num_predict=8192)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    # This line assumes 'rag_retriever' is correctly defined elsewhere and used here as 'retriever'
    # Note: If retriever.retrieve is synchronous, this is fine.
    # The previous code implied 'rag_retriever' was defined globally before this section.
    try:
        results=retriever.retrieve(query,top_k=top_k)
    except NameError:
        print("ERROR: 'rag_retriever' or 'retriever' is not defined. Skipping retrieval.")
        return "Initialization Error: Retriever not available."
    
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using Ollama LLM
    # We define the template as a standard string to keep the structure clear
    prompt_template = """### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to analyze the provided context and answer the user's question following a strict legal assessment structure.

### Guidelines
1. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
2. **Tone:** Maintain a professional, objective legal tone. Use precise terminology suitable for a legal professional.
3. **Citations:** When making claims, reference the specific part of the context (e.g., "According to Article 4...") if available.
4. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question." Do not attempt to make up an answer.

### Response Structure
You must provide your answer in three distinct parts. Do not skip any steps.

**Part 1: Analysis of the Agreement**
Identify and quote the specific sections from the provided agreement or document that are relevant to the question.

**Part 2: Relevant Legal Standards**
Identify and quote the relevant sections from the provided standards or legal materials (e.g., ISO standards, regulations) that apply to the situation.

**Part 3: Legal Assessment**
Based on the evidence gathered in Part 1 and Part 2, provide a clear and professional legal conclusion. Synthesize the information to answer the user's question directly.

### Context
{context}

### Question
{query}

### Answer
"""

    # Format the string with the actual variables
    formatted_prompt = prompt_template.format(context=context, query=query)

    # We use llm.invoke with a list of messages (HumanMessage)
    # Note: HumanMessage requires the 'content' to be a fully formatted string
    response = llm.invoke([HumanMessage(content=formatted_prompt)])
    
    return response.content


In [ ]:
answer=rag_simple("What is Paris Agreement",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Paris Agreement'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering with score >= 0.0)


KeyboardInterrupt: 

## Enhanced RAG

In [ ]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = """### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to analyze the provided context and answer the user's question following a strict legal assessment structure.

### Guidelines
1. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
2. **Tone:** Maintain a professional, objective legal tone. Use precise terminology suitable for a legal professional.
3. **Citations:** When making claims, reference the specific part of the context (e.g., "According to Article 4...") if available.
4. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question." Do not attempt to make up an answer.

### Response Structure
You must provide your answer in three distinct parts. Do not skip any steps.

**Part 1: Analysis of the Agreement**
Identify and quote the specific sections from the provided agreement or document that are relevant to the question.

**Part 2: Relevant Legal Standards**
Identify and quote the relevant sections from the provided standards or legal materials (e.g., ISO standards, regulations) that apply to the situation.

**Part 3: Legal Assessment**
Based on the evidence gathered in Part 1 and Part 2, provide a clear and professional legal conclusion. Synthesize the information to answer the user's question directly.

### Context
{context}

### Question
{query}

### Answer
"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# **Part 3: Legal Assessment**
# Based on the evidence gathered in Part 1 and Part 2, provide a clear and professional legal conclusion. Synthesize the information to answer the user's question directly.


# Example usage:
# result = rag_advanced("Assessing Swiss-Norway agreement on Carbon Dioxide Removal vis-á-vis STAN-METH-OO1 and STAN-METH-002 and RMP (’Article 6 standards’) documents in terms of how the issue of leakage is tackled by the agreement", rag_retriever, llm, top_k=5, min_score=0.1, return_context=True)
# print("Answer:", result['answer'])
# print("Sources:", result['sources'])
# print("Confidence:", result['confidence'])
# print("Context Preview:", result['context'][:300])

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            
            # Streaming answer simulation
            prompt = """### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysis of the provided context to answer the user's question.

### Guidelines
1. **Unrestricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question."

### Response Structure
You must provide your answer in three distinct, detailed parts:

**Part 1: Comprehensive Analysis of the Agreement**
Identify, quote, and analyze *every* specific section from the provided agreement that is relevant to the question. Elaborate on the definitions and specific wording used in the text.

**Part 2: Relevant Legal Standards & Regulations**
Identify and detail the relevant sections from provided standards or legal materials (e.g., ISO, regulations). Explain the specific metrics, compliance requirements, or technical obligations in full.

**Part 3: In-Depth Legal Assessment**
Based on the comprehensive evidence gathered in Parts 1 and 2, provide a lengthy and detailed legal conclusion. Synthesize the information to explore the implications, obligations, and dispute resolution mechanisms fully.

### Context
{context}

### Question
{question}

### Answer
"""
            if stream:
                print("Streaming answer:")
                # We format here temporarily just for the print simulation to look correct
                temp_formatted = prompt.format(context=context, question=question)
                for i in range(0, len(temp_formatted), 80):
                    print(temp_formatted[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer
        
        
        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query('''How are "conservative" estimates defined and used in Article 6 standards and bilateral agreements?''', top_k=28, min_score=0.5, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'How are "conservative" estimates defined and used in Article 6 standards and bilateral agreements?'
Top K: 28, Score threshold: 0.5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.42it/s]

Generated embeddings with shape: (1, 384)
Retrieved 12 documents (after filtering with score >= 0.5)
Streaming answer:
### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysis of the provided context to answer the user's question.

### Guidelines
1. **Un

restricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question."

### Response Structure
You must provide your answer in three disti

In [ ]:

result = adv_rag.query('''Find and list all these usages across different contexts and flag and explain any differences between these definitions or their interpretations''', top_k=28, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
# print("Summary:", result['summary'])
# print("History:", result['history'][-1])



Retrieving documents for query: 'Find and list all these usages across different contexts and flag and explain any differences between these definitions or their interpretations'
Top K: 28, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 28 documents (after filtering with score >= 0.1)
Streaming answer:
### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysis of the provided context to answer the user's question.

### Guidelines
1. **Un

restricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question."

### Response Structure
You must provide your answer in three disti

In [ ]:
result = adv_rag.query('''What are the safeguards that bilateral agreements provide for the additionality of carbon credits?''', top_k=28, min_score=0.2, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])

Retrieving documents for query: 'What are the safeguards that bilateral agreements provide for the additionality of carbon credits?'
Top K: 28, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]

Generated embeddings with shape: (1, 384)
Retrieved 28 documents (after filtering with score >= 0.2)
Streaming answer:
### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysis of the provided context to answer the user's question.

### Guidelines
1. **Un

restricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question."

### Response Structure
You must provide your answer in three disti

In [ ]:
result = adv_rag.query('''List all the provisions regarding additionality in bilateral agreements and flag and explain any differences between these provisions.''', top_k=60, min_score=0.5, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])

Retrieving documents for query: 'List all the provisions regarding additionality in bilateral agreements and flag and explain any differences between these provisions.'
Top K: 60, Score threshold: 0.5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]

Generated embeddings with shape: (1, 384)
Retrieved 36 documents (after filtering with score >= 0.5)
Streaming answer:
### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysis of the provided context to answer the user's question.

### Guidelines
1. **Un

restricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question."

### Response Structure
You must provide your answer in three disti

In [ ]:
result = adv_rag.query(''' What are the safeguards that bilateral agreements provide for the additionality of carbon credits? List all the provisions regarding additionality in bilateral agreements and flag and explain any differences between these provisions.''', top_k=40, min_score=0.4, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])

Retrieving documents for query: ' What are the safeguards that bilateral agreements provide for the additionality of carbon credits? List all the provisions regarding additionality in bilateral agreements and flag and explain any differences between these provisions.'
Top K: 40, Score threshold: 0.4
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 40 documents (after filtering with score >= 0.4)
Streaming answer:
### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysis of the provided context to answer the user's question.

### Guidelines
1. **Un

restricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question."

### Response Structure
You must provide your answer in three disti

In [ ]:
result = adv_rag.query('''list all the countries that have signed the paris agreement''', top_k=350, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
# print("Summary:", result['summary'])
# print("History:", result['history'][-1])


Retrieving documents for query: 'list all the countries that have signed the paris agreement'
Top K: 350, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.25it/s]

Generated embeddings with shape: (1, 384)
Retrieved 350 documents (after filtering with score >= 0.1)
Streaming answer:
### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysi

s of the provided context to answer the user's question.

### Guidelines
1. **Unrestricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer thi

In [ ]:
import numpy

In [ ]:
result = adv_rag.query('''How do countries include the oil and gas sector inn their NDCs''', top_k=30, min_score=0.3, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
# print("Summary:", result['summary'])
# print("History:", result['history'][-1])

Retrieving documents for query: 'How do countries include the oil and gas sector inn their NDCs'
Top K: 30, Score threshold: 0.3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.34s/it]

Generated embeddings with shape: (1, 384)
Retrieved 30 documents (after filtering with score >= 0.3)
Streaming answer:
### Instruction
You are an expert legal assistant specializing in Climate Law. Your task is to conduct an exhaustive, highly detailed, and comprehensive analysis of the provided context to answer the user's question.

### Guidelines
1. **Un

restricted Verbosity:** There are NO constraints on the length of your response. You must be as verbose and thorough as the complexity of the retrieved text demands. Do not summarize, condense, or oversimplify complex legal provisions.
2. **Completeness:** Address every single relevant nuance, sub-clause, exception, and condition found in the text. If the context provides multiple viewpoints or detailed procedures, explain them all in depth.
3. **Source Truth:** Answer using ONLY the information provided in the "Context" section below. Do not use outside knowledge.
4. **Tone:** Maintain a professional, objective, and scholarly legal tone.
5. **Citations:** Rigorously reference specific articles, sections, or paragraphs from the context when making claims.
6. **Fallback:** If the context does not contain the answer, explicitly state: "The provided context does not contain sufficient information to answer this question."

### Response Structure
You must provide your answer in three disti